# ДЗ-2. Витрина клиента

Вы — аналитик ПРАЙМ, и у вас свой **сегмент клиентов**: его прислал бот 26 сентября, он же лежит в вашей строке `prime.assignments`, в колонке `client_filter`. Окно фактов у всех общее — первое полугодие 2026 года.

Задача недели — собрать **витрину клиента**: таблицу, в которой одна строка — один клиент, а рядом всё, что мы про него знаем. Из витрины потом считается всё остальное, поэтому лишние деньги в ней становятся лишними везде.

Идём по лестнице. Сначала вы дописываете один аргумент, как на семинаре, потом целую строку, потом пишете задачу сами, а в конце собираете витрину и доказываете протоколом, что двойного счёта нет.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение, ваш сегмент, выгрузка восьми таблиц | запускаете |
| **2. Уровень 1** | задачи 1–4: не хватает одного аргумента | вписываете аргумент вместо `ЗАПОЛНИТЕ` |
| **3. Уровень 2** | задачи 5–8: не хватает строки | дописываете строку |
| **4. Уровень 3** | задачи 9–12 | пишете код целиком |
| **5. Уровень 4** | задача 13: витрина и протокол сборки | пишете код целиком |
| **6. Итог** | готовый код: ваши ответы и проверка их формата | запускаете перед сдачей |
| **7. Исследование** | две части для тех, кто хочет 9 или 10 | по желанию |

После каждого уровня — вопрос, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N».

**`ЗАПОЛНИТЕ`** — место, куда вписать ваш код. Пока оно не заполнено, ячейка падает с `NameError: name 'ЗАПОЛНИТЕ' is not defined` — так и задумано.

**Оценка.** Части 1–6 дают до 8 баллов: сделали всё аккуратно — 8, и это «отлично». 9 и 10 ставятся только за исследование в части 7. Все критерии и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-2-витрина-клиента).

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей выполните **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в вашем репозитории как `notebooks/hw2.ipynb`, JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно: запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Подключение | подключается к базе через ваш `.env` и заводит функцию `q(sql)` |
| 1.2 Ваш сегмент | достаёт условие `client_filter` из вашей строки `prime.assignments` |
| 1.3 Выгрузка | загружает восемь таблиц — дальше всё в pandas |
| 1.4 Формат ответов | заводит служебную функцию для итога |

Весь SQL задания собран здесь, в готовом коде. Дальше вы работаете только в pandas.

### 1.1 Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает `DataFrame`, как на семинарах. Соединение открывается на каждый запрос и сразу закрывается.

Если здесь ошибка `Пакет prime не найден`, ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN`, в корне `prime-monitor` нет файла `.env`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### 1.2 Ваш сегмент

Сегмент — это клиенты, которые встречаются в вашей таблице с вашим признаком за ваш период. Бот прислал его вам 26 сентября готовым условием для `where`, оно же лежит в вашей строке `prime.assignments`.

**Эту ячейку не меняйте.** При проверке преподаватель подставит сюда условие **другого** сегмента, и все ответы должны пересчитаться сами. Поэтому ниже нигде не вписывайте руками ни условие, ни числа своего сегмента.

In [ ]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

### 1.3 Выгрузка

Восемь таблиц: первые пять — про ваш сегмент, последние три — справочник тарифов и данные всей базы.

| Переменная | Одна строка — это | Что внутри |
|---|---|---|
| `clients` | клиент сегмента | `client_id`, дата регистрации, регион, канал привлечения |
| `subs` | подписка, которой владеет клиент сегмента | тариф, даты начала и конца, пробная ли, причина отмены |
| `pay` | попытка списания по этим подпискам за окно фактов | сумма, **статус**, способ оплаты |
| `sub_members` | участник подписки из `subs` | `subscription_id` и `member_id` — кто подключён к подписке |
| `members` | участие клиента сегмента в любой подписке | `subscription_id` и `client_id` — клиент сегмента как участник |
| `tariffs` | цена тарифа в интервале дат | весь справочник, пять строк |
| `spend` | расходы канала за день | `channel`, `spend_date`, `amount` |
| `registrations` | канал × день регистрации | `n_clients` — сколько клиентов всей базы пришло из канала в этот день |

**Окно фактов общее для всех:** с 1 января по 30 июня 2026 года включительно. В запросе правый конец не включён: `paid_at < '2026-07-01'`.

Самые большие сегменты выгружаются около минуты и занимают в памяти до 2 ГБ — закройте лишние вкладки браузера. Если выгрузка идёт дольше двух минут, сервер оборвёт запрос — что делать, написано в тексте задания, раздел «Если что-то не работает».

In [ ]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включён
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

### 1.4 Формат ответов

Служебная функция: по ней итог в части 6 проверяет, что ответ записан в нужном формате — целое, число с копейками, список. **Верно ли посчитано, она не знает и не сообщает**: это проверяет преподаватель. Код ячейки свёрнут, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 2 знаков — рубли",
    2: "int",
    3: "int",
    4: "float до 2 знаков — рубли (0.0, если таких клиентов нет)",
    5: "float до 2 знаков — рубли",
    6: "float до 2 знаков — рубли",
    7: "int",
    8: "float до 1 знака — проценты",
    9: "int",
    10: "list [float до 2 знаков, int]",
    11: "list [float до 2 знаков, int]",
    12: "list из 3 int",
    13: "list из 3 int — то, что вернула print_protocol()",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:   # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def pct(x) -> bool:     # float, у которого не больше одного знака после запятой
        return type(x) is float and round(x, 1) == x

    def whole(x) -> bool:
        return type(x) is int

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: money, 2: whole, 3: whole, 4: money, 5: money, 6: money, 7: whole, 8: pct,
        9: whole,
        10: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        11: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        12: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
        13: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Уровень 1 — дыра в один аргумент

Как на семинаре: код готов, не хватает одного аргумента. Впишите его вместо `ЗАПОЛНИТЕ` и запустите ячейку.

Последняя строка каждой задачи кладёт ответ в `answer_N`. Эти строки и имена `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом сегменте.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All. Замените всё содержимое ячейки одной строкой `answer_N = None`, где N — номер задачи: итог покажет «❌ не решена», а остальное проверится как обычно.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 6 проверяет только формат. Правильность проверяет преподаватель — на вашем сегменте и на контрольном.

**В `pay` лежат все попытки списания:** `success`, `failed` и `refunded`. Какие из них деньги, решаете вы — это задача 1.

### Задача 1. Контрольное число ДЕНЬГИ

Первое правило сборки витрины: контрольное число считают до сборки. Мы заранее знаем, сколько денег принёс сегмент за окно, — и потом витрина обязана дать ровно столько же.

**Что сделать**

Оставьте в `pay` только платежи, которые действительно принесли деньги, и посчитайте их сумму.

**Формат ответа:** `float` до двух знаков, рубли — например `98765432.1`.

**Подсказка:** Какие статусы бывают, покажет `pay["status"].value_counts()`.

In [ ]:
# Какие попытки списания — это деньги, которые остались у компании?
success = pay[pay["status"] == 'success']
revenue = success["amount"].sum()

print("платежей с этим статусом:", len(success))
answer_1 = round(float(revenue), 2)
answer_1



### Задача 2. Подписки без платежа

Руководитель спрашивает, сколько подписок сегмента за полгода не принесли ни рубля. Чтобы ответить, к каждой подписке присоединяют число её успешных платежей и смотрят, у кого пусто.

**Что сделать**

Выберите вид соединения так, чтобы подписки без платежей **остались** в результате — с пропуском в `n_payments`.

**Формат ответа:** `int` — например `4321`.

**Подсказка:** Слайд 7 Л4: `how=` решает судьбу строк, которым не нашлась пара.

In [ ]:
# Сколько успешных платежей пришло по каждой подписке за окно.
by_sub = (pay[pay["status"] == "success"]
          .groupby("subscription_id", as_index=False)
          .agg(n_payments=("payment_id", "size")))

# Подписка без платежей должна остаться в результате — с пропуском в n_payments.
subs_pay = subs.merge(by_sub, on="subscription_id", how='left', validate="1:1")

no_payment = subs_pay["n_payments"].isna().sum()
print("подписок:", len(subs), " строк после соединения:", len(subs_pay))
answer_2 = int(no_payment)
answer_2

### Задача 3. Кто платил

Сколько владельцев из сегмента заплатили за полгода хотя бы раз? Для этого к каждому платежу приписывают владельца подписки. Но прежде чем соединять, объявляют кратность: сколько платежей приходится на одну подписку и сколько подписок — на один платёж.

**Что сделать**

Впишите в `validate=` кратность связи «платёж → подписка».

**Формат ответа:** `int` — например `5432`.

**Подсказка:** Слайд 6 Л4, четыре случая кратности. Если слева на один ключ приходится много строк, это «m».

In [ ]:
success = pay[pay["status"] == "success"]

# Сколько платежей приходится на одну подписку — и сколько подписок на один платёж?
pay_owner = success.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id", how="left", validate="m:1",
)

payers = pay_owner["owner_client_id"].nunique()
print("платежей до соединения:", len(success), " после:", len(pay_owner))
answer_3 = int(payers)
answer_3

### Задача 4. Выручка без канала

Маркетинг смотрит выручку по каналам привлечения. У части клиентов канал не записан, и если этого не заметить, сумма по каналам тихо разойдётся с общей выручкой.

**Что сделать**

Сделайте так, чтобы клиенты без канала попали в отчёт отдельной строкой, и посчитайте их выручку.

**Формат ответа:** `float` до двух знаков, рубли — например `123456.7`. Если таких клиентов в сегменте нет — `0.0`.

**Подсказка:** Слайд 18 Л4: сумма по группам не сходится с общей. Сверьте сумму по каналам с выручкой задачи 1.

In [ ]:
success = pay[pay["status"] == "success"]
pay_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="m:1"))

# Клиенты без канала должны остаться в отчёте отдельной строкой.
by_channel = pay_client.groupby("acquisition_channel", dropna=False)["amount"].sum()

print(by_channel.round(2).to_string())
no_channel = by_channel[by_channel.index.isna()].sum()
answer_4 = round(float(no_channel), 2)
answer_4

**Вопрос 1.** В `pay` есть попытки списания со статусом `refunded`, и сумма у них положительная. Сколько таких строк в вашем сегменте и на какую сумму? Почему эту сумму нельзя прибавить к выручке — и почему её нельзя вычесть из выручки?

In [ ]:
refunded_payments = pay[pay["status"] == "refunded"]
count_refunded = len(refunded_payments)
sum_refunded = refunded_payments["amount"].sum()

print(f"Количество строк со статусом 'refunded': {count_refunded}")
print(f"Сумма этих платежей: {round(float(sum_refunded), 2)}")


#### ✍️ Ответ на вопрос 1

Количество строк: 167; на сумму: 53233.0. Данную сумму нельзя прибавить к выручке, так как это отток денег (возврат средств клиенту). Нельзя убавить по причине того, что платежи со статусом 'refunded' уже не входят в сумму выручки, ранее были отфильтрованы, при вычитание получим двойное занижение выручки.

## Часть 3. Уровень 2 — дыра в строку

Теперь не хватает целой строки: выражения, условия или набора аргументов. Всё остальное готово. Правила те же: строки с `answer_N` не трогайте, не получилось — `answer_N = None`.

### Задача 5. Свернуть до клиента

Средний доход с платящего клиента — ARPPU — это выручка, делённая на число клиентов, которые платили. Чтобы посчитать его по витрине, платежи сначала сворачивают до зерна «клиент» и только потом соединяют с клиентами — иначе соединение перемножит строки.

**Что сделать**

Допишите `agg(...)`: для каждого владельца — `n_payments` (сколько успешных платежей) и `paid` (сколько он заплатил).

**Формат ответа:** `float` до двух знаков, рубли — например `1234.56`.

**Подсказка:** Слайд 17 Л4, именованные агрегаты: `agg(имя=("колонка", "функция"), ...)`.

In [ ]:
success = pay[pay["status"] == "success"]

# Одна строка — один владелец: сколько успешных платежей и сколько денег.
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id", "size"), paid=("amount", "sum")))

mart5 = clients.merge(by_client, left_on="client_id", right_on="owner_client_id",
                      how="left", validate="1:1")

paying = mart5["n_payments"].notna().sum()
arppu = mart5["paid"].sum() / paying
print("клиентов:", len(mart5), " из них платили:", paying)
answer_5 = round(float(arppu), 2)
answer_5

### Задача 6. Цена по справочнику

В `tariffs` лежит история цен: у тарифа несколько строк с интервалами действия, а одна строка ошибочная — вы нашли её на семинаре. Сколько принёс тариф «Расширенный» по цене справочника?

**Что сделать**

Допишите две строки: 1) справочник без ошибочной строки; 2) условие «момент платежа попадает в интервал действия цены» по конвенции курса.

**Формат ответа:** `float` до двух знаков, рубли — например `2345678.9`.

**Подсказка:** Раздел 2 семинара С4. Конвенция границы — `(paid_at >= valid_from) & (paid_at < valid_to + 1 день)`, потому что `valid_to` — это дата, то есть полночь. Каждый платёж должен найти в справочнике ровно одну цену: сравните число строк до и после.

In [ ]:
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
ref_fixed = ref[~(ref['valid_from'] < ref.groupby('tariff_id')['valid_to'].shift(1))]

pay_t = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
m = pay_t.merge(ref_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]],
                on="tariff_id")

# 2. Момент платежа попадает в интервал действия цены — по конвенции курса.
hit = (m['paid_at'] >= m['valid_from']) & (m['paid_at'] < m['valid_to'] + pd.Timedelta(days=1))
priced = m[hit]

print("платежей:", len(pay_t), " строк с ценой:", len(priced))
extended = priced.loc[priced["tariff_name"] == "Расширенный", "monthly_fee"].sum()
answer_6 = round(float(extended), 2)
answer_6

### Задача 7. Выше среднего по своему каналу

Маркетинг хочет знать, сколько платящих клиентов приносят больше, чем типичный клиент **их же** канала. Для этого рядом с каждым клиентом ставят среднее по его каналу — не сворачивая таблицу.

**Что сделать**

Допишите строку: `channel_avg` — среднее `paid` по каналу клиента, по значению в каждой строке.

**Формат ответа:** `int` — например `6543`.

**Подсказка:** Слайд 19 Л4: `transform` возвращает столько же строк, сколько было.

In [ ]:
success = pay[pay["status"] == "success"]
per_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .groupby("owner_client_id", as_index=False)
              .agg(paid=("amount", "sum"))
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="1:1"))
per_client = per_client[per_client["acquisition_channel"].notna()]   # только известные каналы

# Рядом с каждым клиентом — среднее paid по его каналу. Строк остаётся столько же.
per_client["channel_avg"] = per_client.groupby("acquisition_channel")["paid"].transform("mean")

above = (per_client["paid"] > per_client["channel_avg"]).sum()
print("платящих клиентов с известным каналом:", len(per_client))
answer_7 = int(above)
answer_7

### Задача 8. Выручка по месяцам

Финансам нужна таблица: строки — месяцы, колонки — тарифы, в клетках — выручка. По ней видно, как выручка «Базового» менялась от месяца к месяцу.

**Что сделать**

Допишите аргументы `pivot_table`: строки — `month`, колонки — `tariff_id`, значения — сумма `amount`.

**Формат ответа:** `float` до одного знака, проценты — например `87.6`.

**Подсказка:** Слайд 23 Л4.

In [ ]:
success = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# Строки — месяцы, колонки — тарифы (1 — «Базовый», 2 — «Расширенный»), в клетках — выручка.
by_month = success.pivot_table(index="month", columns="tariff_id", values="amount", aggfunc="sum")

print(by_month.round(2).to_string())
april_to_march = by_month.loc["2026-04", 1] / by_month.loc["2026-03", 1] * 100
answer_8 = round(float(april_to_march), 1)
answer_8

**Вопрос 2.** Выручка «Базового» в апреле — `answer_8` процентов от мартовской. Это отток клиентов? Проверьте по дням: сколько дней апреля есть в `pay` и как выглядит выручка по дням. Объясните, что случилось, и что вы написали бы финансам в отчёте.

In [ ]:
success = pay[pay["status"] == "success"]
base_payments = success.merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1")
base_payments = base_payments[base_payments["tariff_id"] == 1].copy()

base_payments["date"] = pd.to_datetime(base_payments["paid_at"]).dt.date
base_payments["month_str"] = pd.to_datetime(base_payments["paid_at"]).dt.strftime("%Y-%m")

days_march = base_payments[base_payments["month_str"] == "2026-03"]["date"].nunique()
days_april = base_payments[base_payments["month_str"] == "2026-04"]["date"].nunique()

print(f"Дней с платежами в марте: {days_march}")
print(f"Дней с платежами в апреле: {days_april}")

daily_rev = base_payments.groupby(["month_str", "date"])["amount"].sum().reset_index()
avg_march = daily_rev[daily_rev["month_str"] == "2026-03"]["amount"].mean()
avg_april = daily_rev[daily_rev["month_str"] == "2026-04"]["amount"].mean()

print(f"Средняя выручка за день в марте: {avg_march:.2f}")
print(f"Средняя выручка за день в апреле: {avg_april:.2f}")

#### ✍️ Ответ на вопрос 2

Март: 31 день, средняя выручка за день: 11269.26; Апрель: 28 дней, средняя выручка за день: 10760.36. Средняя выручка в апреле ниже, чем в марте, но это не показатель оттока клиентов. Скорее всего причина в количистве дней каждого месяца. Для отчета я бы урезала дни марта до 28(убрала 3 последних), пересчитала бы средний показатель и сравнила бы с апрелем. Сейчас данные не дают точного ответа.

## Часть 4. Уровень 3 — пишете сами

Заготовок больше нет: в ячейке только условие в комментарии и строка с `answer_N`. Код вы пишете целиком — от загруженных таблиц до ответа. Печатайте промежуточные результаты: так проще найти ошибку и объяснить решение на защите.

Каждая задача опирается на приём с лекции Л4 и семинара С4 — номер слайда стоит в подсказке. Ответ кладите в `answer_N` обычным `int` или `float`, а не `numpy`-числом: `int(...)`, `round(float(...), 2)`.

### Задача 9. Люди, а не места

Сколько людей сегмента пользуются ПРАЙМ? Пользуется тот, кто владеет подпиской или участвует хотя бы в одной чужой. Один человек может участвовать в нескольких подписках: если не свернуть участия до клиента, вы насчитаете места, а не людей.

**Что сделать**

Посчитайте клиентов сегмента, которые владеют подпиской (`subs`) или участвуют хотя бы в одной (`members`). Одна строка — один клиент.

**Формат ответа:** `int` — например `7654`.

**Подсказка:** Раздел 3 семинара С4: свернуть участия до клиента, соединить с `validate="1:1"`, флаг «владелец или участник».

In [ ]:
owners = subs[['owner_client_id']].drop_duplicates().rename(columns={"owner_client_id": "client_id"})
participants = members[["client_id"]].drop_duplicates()

merged = owners.merge(participants, on="client_id", how="outer", validate="1:1", indicator=True)
answer_9 = int(len(merged))

print("Всего уникальных клиентов ПРАЙМ:", answer_9)

answer_9

### Задача 10. Сколько человек на подписку

Подпиской пользуются владелец и те, кого он подключил. Подключить можно не больше `share_slots` человек — это поле справочника тарифов. Сколько в среднем человек приходится на подписку сегмента и сколько подписок заполнены до предела?

**Что сделать**

1. Для каждой подписки из `subs`: один владелец плюс число её строк в `sub_members`.
2. Среднее по всем подпискам сегмента — первая часть ответа.
3. Присоедините к подпискам `share_slots` из `tariffs`. Подписок, где людей ровно `share_slots` + 1, — вторая часть ответа.
4. Напечатайте, у скольких подписок людей больше, чем `share_slots` + 1. Это инвариант: число обязано быть нулём.

**Формат ответа:** `[среднее, заполненных до предела]` — например `[3.21, 987]`: `float` до двух знаков и `int`.

**Подсказка:** Слайды 8 и 14 Л4. У одного `tariff_id` в справочнике несколько строк, и `validate="m:1"` об этом скажет. Какая часть справочника на самом деле нужна для `share_slots`?

In [ ]:
member_counts = (members
                 .groupby("subscription_id", as_index=False)
                 .agg(num_members=("client_id", "size")))

subs_people = subs.merge(member_counts, on="subscription_id",
                         how="left", validate="1:1")

subs_people["num_members"] = subs_people["num_members"].fillna(0).astype(int)
subs_people["total_people"] = subs_people["num_members"] + 1

avg_people = subs_people["total_people"].mean()
print(f"Среднее людей на подписку: {round(float(avg_people), 2)}")

tariffs_slots = tariffs[["tariff_id", "share_slots"]].drop_duplicates(subset=["tariff_id"])

subs_limits = subs_people.merge(tariffs_slots, on="tariff_id",
                                how="left", validate="m:1")

subs_limits["capacity"] = subs_limits["share_slots"] + 1

filled_to_limit = int((subs_limits["total_people"] == subs_limits["capacity"]).sum())
print(f"Подписок, заполненных до предела: {filled_to_limit}")

over_limit = int((subs_limits["total_people"] > subs_limits["capacity"]).sum())
print(f"Подписок, превышающих лимит (инвариант, ожидание 0): {over_limit}")

answer_10 = [round(float(avg_people), 2), filled_to_limit]
print("Ответ:", answer_10)

answer_10

### Задача 11. Сколько стоил клиент

CAC — сколько стоило привести одного клиента. Расходы на привлечение известны по каналу и дню, а не по клиенту: присоединить их к витрине напрямую нельзя. Поэтому CAC считают на его собственном зерне — «канал × месяц» — и только потом приписывают клиентам.

**Что сделать**

1. CAC канала за месяц — это расходы канала за месяц (`spend`), делённые на число всех клиентов базы, которые пришли из этого канала в этом месяце (`registrations`). Берите только полные месяцы расходов — с 2025-02 по 2026-07.
2. Каждому клиенту сегмента припишите CAC его канала в месяц его регистрации.
3. Посчитайте средний CAC по клиентам сегмента, у которых он определён, и число клиентов, у которых он не определён: канал не записан или регистрация вне этих месяцев.

**Формат ответа:** `[средний CAC, клиентов без CAC]` — например `[432.1, 876]`: `float` до двух знаков и `int`.

**Подсказка:** Слайд 28 Л4: `marketing_spend` не присоединяется к витрине — у него другое зерно. Месяц из даты: `pd.to_datetime(...).dt.strftime("%Y-%m")`. Соединение по двум колонкам: `on=["channel", "month"]`, и кратность объявите.

In [ ]:
clients_all = q("""
    SELECT client_id, acquisition_channel, registered_at 
    FROM prime.clients
""")

clients_all['reg_month'] = pd.to_datetime(clients_all['registered_at']).dt.strftime("%Y-%m")

clients_valid = clients_all[
    (clients_all['reg_month'] >= '2025-02') & 
    (clients_all['reg_month'] <= '2026-07')
]

registrations = (
    clients_valid
    .groupby(['acquisition_channel', 'reg_month'], as_index=False)
    .size()
    .rename(columns={'size': 'registrations'})
)

marketing_spend = q("""
    SELECT channel, 
           spend_date AS date, 
           amount AS spend
    FROM prime.marketing_spend
""")

marketing_spend['month'] = pd.to_datetime(marketing_spend['date']).dt.strftime("%Y-%m")

spend_monthly = (
    marketing_spend[
        (marketing_spend['month'] >= '2025-02') & 
        (marketing_spend['month'] <= '2026-07')
    ]
    .groupby(['channel', 'month'], as_index=False)['spend']
    .sum()
)

cac_table = spend_monthly.merge(
    registrations,
    left_on=['channel', 'month'],
    right_on=['acquisition_channel', 'reg_month'],
    how='inner'
)

cac_table['CAC'] = cac_table['spend'] / cac_table['registrations']

segment_clients = clients.copy()  
segment_clients['reg_month'] = pd.to_datetime(segment_clients['registered_at']).dt.strftime("%Y-%m")

merged = segment_clients.merge(
    cac_table[['channel', 'month', 'CAC']],
    left_on=['acquisition_channel', 'reg_month'],
    right_on=['channel', 'month'],
    how='left',
    validate="m:1" 
)

clients_no_CAC = int(merged['CAC'].isna().sum())

mean_cac = round(float(merged['CAC'].mean()), 2)

answer_11 = [mean_cac, clients_no_CAC]

print("Ответ:", answer_11)

answer_11



### Задача 12. Лучшие регионы канала

Где продавать ПРАЙМ дальше? Маркетинг просит три лучших региона по выручке в каждом канале привлечения — и начинает с самого доходного канала.

**Что сделать**

Посчитайте выручку сегмента за окно по каналу и региону владельца; клиенты без канала не участвуют. В каждом канале отберите три региона с наибольшей выручкой. Ответ — три `region_code` самого доходного канала сегмента, от большей выручки к меньшей.

**Формат ответа:** список из трёх `int` — например `[12, 34, 56]`.

**Подсказка:** Слайд 25 Л4: «три лучших вообще» и «три лучших в каждом канале» — разные вещи.

In [ ]:
success_pay = pay[pay['status'] == 'success']

pay_owner = success_pay.merge(
    subs[['subscription_id', 'owner_client_id']],
    on='subscription_id',
    validate='m:1'
)

pay_client = pay_owner.merge(
    clients[['client_id', 'acquisition_channel', 'region_code']],
    left_on='owner_client_id',
    right_on='client_id',
    validate='m:1'
)

pay_client_clean = pay_client[pay_client['acquisition_channel'].notna()]

rev_by_ch_reg = pay_client_clean.groupby(
    ['acquisition_channel', 'region_code'], 
    as_index=False
)['amount'].sum()

best_channel = rev_by_ch_reg.groupby('acquisition_channel')['amount'].sum().idxmax()

top_3_regions = (
    rev_by_ch_reg[rev_by_ch_reg['acquisition_channel'] == best_channel]
    .sort_values('amount', ascending=False)
    .head(3)
)

answer_12 = top_3_regions['region_code'].astype(int).tolist()

print(f"Самый доходный канал: {best_channel}")
print("Топ-3 региона этого канала:", answer_12)

answer_12

**Вопрос 3.** Куда вы добавили бы рубль на привлечение? Сравните каналы по двум числам: CAC канала (задача 11) и выручка на клиента вашего сегмента из этого канала. Назовите канал и оба числа. Почему выручку вашего сегмента нельзя просто разделить на CAC канала — какую оговорку нужно сделать?

In [ ]:
total_spend = spend_monthly.groupby('channel')['spend'].sum()
total_regs = registrations.groupby('acquisition_channel')['registrations'].sum()
global_cac = (total_spend / total_regs).dropna().rename('CAC_канала')

pay_with_owner = success.merge(
    subs[['subscription_id', 'owner_client_id']], 
    on='subscription_id', 
    validate='m:1'
)

pay_by_client = pay_with_owner.groupby('owner_client_id', as_index=False).agg(
    revenue=('amount', 'sum')
)

clients_with_revenue = clients.merge(
    pay_by_client, 
    left_on='client_id', 
    right_on='owner_client_id', 
    how='left',
    validate='1:1'
)
clients_with_revenue['revenue'] = clients_with_revenue['revenue'].fillna(0)

seg_stats = clients_with_revenue.groupby('acquisition_channel').agg(
    total_revenue=('revenue', 'sum'),
    clients_count=('client_id', 'count')
)
seg_stats['Выручка_на_клиента'] = seg_stats['total_revenue'] / seg_stats['clients_count']

comparison = pd.concat([global_cac, seg_stats['Выручка_на_клиента']], axis=1).dropna()
comparison['Прокси_LTV/CAC'] = comparison['Выручка_на_клиента'] / comparison['CAC_канала']

comparison = comparison.sort_values('Прокси_LTV/CAC', ascending=False)
print("Сравнение каналов:")
display(comparison.round(2))

best_channel = comparison.index[0]
best_cac = round(comparison.loc[best_channel, 'CAC_канала'], 2)
best_rev = round(comparison.loc[best_channel, 'Выручка_на_клиента'], 2)

print(f"\nЛучший канал для инвестиций: {best_channel}")
print(f"CAC канала: {best_cac} ₽")
print(f"Выручка на клиента в сегменте: {best_rev} ₽")

#### ✍️ Ответ на вопрос 3
Я бы добавила "рубль" на привлечение в "Лучший канал для дополнительных инвестиций: **organic** (CAC: **137.45 ₽**, Выручка на клиента в сегменте: **886.69 ₽**)".
Выручку данного сегмента нельзя просто разделить на глобальный CAC канала, важная оговорка: CAC в Задаче 11 рассчитан по всей базе клиентов, а данный сегмент отбирался по обращениям в поддержку. Значит, что реальная стоимость привлечения именно таких клиентов может значительно отличаться от средней по каналу. Также считаю выручку только за 6 месяцев окна фактов, а не весь жизненный цикл (LTV). Следовательно полученное соотношение является краткосрочной прокси-оценкой окупаемости (не точное значение LTV/CAC). Сам канал с низкой выручкой за полугодичный период может оказаться самым прибыльным в долгосрочной перспективе.

## Часть 5. Уровень 4 — витрина и протокол сборки

Главная задача недели. Вы собираете **витрину клиента**: одна строка — один клиент сегмента, рядом всё, что про него известно. И доказываете, что двойного счёта нет, **протоколом сборки** — той же формы, что на слайде 15 Л4 и на семинаре С4.

> **Протокол — это 3 балла из 10.** Без него ноль, даже если все числа верные: результат, который нельзя перепроверить, в работе не принимают.

Три последние строки протокола — инварианты витрины:

- **ЗЕРНО:** строк в витрине столько же, сколько уникальных клиентов, и столько, сколько вы объявили до сборки;
- **ДЕНЬГИ:** сумма по витрине сходится с независимым контролем до копейки;
- **ЛЮДИ:** число тех, кто пользуется ПРАЙМ, сходится со счётом, сделанным другим путём.

Ниже готовые функции, эту ячейку просто запустите. `step()` — та же, что на семинаре: одна строка протокола на одно соединение. `check_step()` делает строку протокола для проверки из вашего `src/prime/checks.py`. `print_protocol()` печатает протокол и возвращает ответ задачи 13.

In [ ]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


def check_step(name: str, frame: pd.DataFrame, bad: pd.DataFrame, expected: int) -> dict:
    """Строка протокола для проверки из checks.py.

    bad — строки, которые нашла проверка; expected — сколько вы ждали, объявив это ДО сборки.
    """
    found = len(bad)
    verdict = f"ok, ожидали {expected}" if found == expected else f"ВНИМАНИЕ: нашлось {found}, ждали {expected}"
    return {"step": name, "rows_before": len(frame), "rows_after": len(frame),
            "validate": "проверка", "keys_lost": found, "verdict": verdict}


def print_protocol(protocol: list[dict], mart: pd.DataFrame, declared: int,
                   money_mart: float, money_control: float,
                   people_mart: int, people_control: int) -> list[int]:
    """Напечатать протокол сборки. Вернуть [строк в витрине, разница ДЕНЬГИ в копейках, ЛЮДИ по витрине]."""
    rows, unique = len(mart), int(mart["client_id"].nunique())
    kopecks_mart, kopecks_control = round(money_mart * 100), round(money_control * 100)
    diff = kopecks_mart - kopecks_control

    print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента")
    print()
    print(pd.DataFrame(protocol).to_string(index=False))
    print()
    print(f"ЗЕРНО   строк {rows}, клиентов {unique}, объявляли {declared}"
          f"   -> {'ok' if rows == unique == declared else 'ВНИМАНИЕ'}")
    print(f"ДЕНЬГИ  по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽, разница {diff} коп."
          f"   -> {'ok' if diff == 0 else 'ВНИМАНИЕ'}")
    print(f"ЛЮДИ    по витрине {people_mart}, независимо {people_control}"
          f"   -> {'ok' if people_mart == people_control else 'ВНИМАНИЕ'}")
    print()
    print("HW2_PROTOCOL=" + json.dumps({
        "steps": protocol, "grain": [rows, unique, declared],
        "money_kopecks": [kopecks_mart, kopecks_control], "people": [int(people_mart), int(people_control)],
    }, ensure_ascii=False, default=str))
    return [int(rows), int(diff), int(people_mart)]


print("функции протокола готовы")

### Задача 13. Витрина клиента

Соберите витрину по своему сегменту и докажите протоколом, что двойного счёта нет. Весь раздел 4 семинара С4 — это тот же путь на общем сегменте.

**Что сделать**

1. **До сборки** объявите, сколько строк будет в витрине, и сколько ключей потеряет каждое левое соединение — эти числа пойдут в `expect_lost`.
2. Соберите `mart`, одна строка — один клиент сегмента:
   - `client_id`, `acquisition_channel`, `region_code` — из `clients`;
   - `subscription_id`, `tariff_id` — подписка клиента, пропуск, если её нет;
   - `n_payments`, `paid` — успешные платежи за окно, свёрнутые до клиента; `0`, если их нет;
   - `n_memberships` — в скольких подписках клиент участник; `0`, если ни в одной;
   - `uses_prime` — владелец или участник.
3. Каждое соединение запишите в протокол через `step()`.
4. Примените к витрине функцию из своего `src/prime/checks.py` — например, `find_missing_channel` — и запишите её строкой `check_step()` с ожиданием, объявленным до сборки. Это отдельные 0,5 балла из 10.
5. Посчитайте независимые контроли для ДЕНЬГИ и ЛЮДИ **не по витрине, а другим путём** и вызовите `print_protocol()`.

**Формат ответа:** то, что вернула `print_protocol()`: список из трёх `int`.

**Подсказка:** Если `src/prime/checks.py` у вас нет — создайте его по образцу семинара С3 и закоммитьте в `main`: `from prime.checks import find_missing_channel`. Протокол должен уметь покраснеть: проверьте его, сломав одно соединение, как на семинаре.

In [ ]:
declared = len(clients)

subs_one = (
    subs.sort_values('started_at')
    .groupby('owner_client_id', as_index=False)
    .tail(1)[['owner_client_id', 'subscription_id', 'tariff_id']]
)

success = pay[pay['status'] == 'success']
pay_by_client = (
    success
    .merge(subs[['subscription_id', 'owner_client_id']], on='subscription_id', validate='m:1')
    .groupby('owner_client_id', as_index=False)
    .agg(n_payments=('payment_id', 'size'), paid=('amount', 'sum'))
)

memberships_by_client = (
    members
    .groupby('client_id', as_index=False)
    .agg(n_memberships=('subscription_id', 'size'))
)

owners = subs[['owner_client_id']].drop_duplicates().rename(columns={'owner_client_id': 'client_id'})
participants = members[['client_id']].drop_duplicates()
prime_users = pd.concat([owners, participants]).drop_duplicates()
prime_users['uses_prime'] = True

expected_lost_subs = int(clients['client_id'].isin(subs_one['owner_client_id']).eq(False).sum())
expected_lost_pay = int(clients['client_id'].isin(pay_by_client['owner_client_id']).eq(False).sum())
expected_lost_members = int(clients['client_id'].isin(memberships_by_client['client_id']).eq(False).sum())
expected_lost_prime = int(clients['client_id'].isin(prime_users['client_id']).eq(False).sum())

print(f"Объявления ДО сборки:")
print(f"  Зерно: {declared} строк")
print(f"  Ожидаем потерь при join с подписками: {expected_lost_subs}")
print(f"  Ожидаем потерь при join с платежами: {expected_lost_pay}")
print(f"  Ожидаем потерь при join с участиями: {expected_lost_members}")
print(f"  Ожидаем потерь при join с uses_prime: {expected_lost_prime}")

protocol = []

mart = clients.copy()

before = mart
mart = mart.merge(subs_one, left_on='client_id', right_on='owner_client_id', how='left', validate='1:1')
mart = mart.drop(columns=['owner_client_id'])
protocol.append(step("подписки", before, mart, key='subscription_id', validate='1:1', expect_lost=expected_lost_subs))

before = mart
mart = mart.merge(pay_by_client, left_on='client_id', right_on='owner_client_id', how='left', validate='1:1')
mart = mart.drop(columns=['owner_client_id'])
protocol.append(step("платежи", before, mart, key='n_payments', validate='1:1', expect_lost=expected_lost_pay))
mart['n_payments'] = mart['n_payments'].fillna(0).astype(int)
mart['paid'] = mart['paid'].fillna(0)

before = mart
mart = mart.merge(memberships_by_client, on='client_id', how='left', validate='1:1')
protocol.append(step("участия", before, mart, key='n_memberships', validate='1:1', expect_lost=expected_lost_members))
mart['n_memberships'] = mart['n_memberships'].fillna(0).astype(int)

before = mart
mart = mart.merge(prime_users, on='client_id', how='left', validate='1:1')
protocol.append(step("uses_prime", before, mart, key='uses_prime', validate='1:1', expect_lost=expected_lost_prime))
mart['uses_prime'] = mart['uses_prime'].fillna(False)

try:
    from prime.checks import find_missing_channel
    bad = find_missing_channel(mart)
    expected_bad = int(clients['acquisition_channel'].isna().sum())
    protocol.append(check_step("проверка канала", mart, bad, expected_bad))
    print(f"\nПроверка из checks.py: найдено {len(bad)} строк без канала, ожидали {expected_bad}")
except ImportError:
    print("\nprime.checks не найден. Определяю локально.")
    def find_missing_channel(df):
        return df[df['acquisition_channel'].isna()]
    bad = find_missing_channel(mart)
    expected_bad = int(clients['acquisition_channel'].isna().sum())
    protocol.append(check_step("проверка канала (локально)", mart, bad, expected_bad))
    print(f"Локальная проверка: найдено {len(bad)} строк без канала, ожидали {expected_bad}")

money_control = answer_1
money_mart = float(mart['paid'].sum())

people_control = answer_9
people_mart = int(mart['uses_prime'].sum())

print(f"\nНезависимые контроли:")
print(f"  ДЕНЬГИ: по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽")
print(f"  ЛЮДИ: по витрине {people_mart}, независимо {people_control}")

answer_13 = print_protocol(protocol, mart, declared, money_mart, money_control, people_mart, people_control)
answer_13

**Вопрос 4.** Где ваш протокол заранее ждал потерь ключей и почему это не ошибка — назовите числа. Что стало бы со строками ЗЕРНО и ЛЮДИ, если бы участия не свернули до клиента? Посчитайте, а не угадывайте.

In [ ]:
print("ГДЕ ПРОТОКОЛ ЖДАЛ ПОТЕРЬ КЛЮЧЕЙ:")

print(f"  • Подписки: 1769 клиентов без подписки")
print(f"  • Платежи: 3365 клиентов без успешных платежей")
print(f"  • Участия: 1900 клиентов без участий в чужих подписках")
print(f"  • uses_prime: 543 клиентов без флага (не владелец и не участник)")
print("Это не ошибка, а особенность данных: не все клиенты сегмента")
print("совершали эти действия в окне фактов.")

print("\nЧТО БЫЛО БЫ БЕЗ СВЁРТКИ УЧАСТИЙ:")

clients_count = len(clients)
print(f"  • Клиентов в сегменте (ЗЕРНО должно быть): {clients_count}")

members_count = len(members)
unique_clients_in_members = members['client_id'].nunique()

print(f"  • Всего строк в members (участий): {members_count}")
print(f"  • Уникальных клиентов среди участников: {unique_clients_in_members}")

mart_without_rollup = clients.merge(members, on='client_id', how='left', validate='1:m')
rows_without_rollup = len(mart_without_rollup)
unique_clients_without_rollup = mart_without_rollup['client_id'].nunique()

print(f"  Если бы не свернули:")
print(f"  • ЗЕРНО (строк в витрине): {rows_without_rollup} (вместо {clients_count})")
print(f"  • ЛЮДИ (уникальных клиентов): {unique_clients_without_rollup}")

uses_prime_without_rollup = mart_without_rollup['client_id'].notna().sum()
print(f"  • ЛЮДИ (сумма флага uses_prime): {uses_prime_without_rollup} (вместо 5345)")

print("\nВЫВОД:")
print(f"  • ЗЕРНО покраснела бы: {rows_without_rollup} строк вместо {clients_count}")
print(f"    (рост на {rows_without_rollup - clients_count} строк из-за дублирования)")
print(f"  • ЛЮДИ покраснела бы: {uses_prime_without_rollup} вместо 5345")
print(f"    (завышение на {uses_prime_without_rollup - 5345} из-за подсчёта мест, а не людей)")

#### ✍️ Ответ на вопрос 4
Протокол заранее ожидал потерю ключей при левых соединениях. Без подписки - 1769 клиентов, без успешных платежей - 3365 ,без участий - 1900  и без флага uses_prime - 543 . Это особенность данных, так как не все клиенты сегмента совершали эти действия в окне фактов. Если бы участия не свернули до клиента, при соединении с таблицей members (6618 строк) клиенты с несколькими подписками продублировались бы, строка ЗЕРНО покраснела бы (строк стало бы больше объявленных 5888), а строка ЛЮДИ тоже покраснела бы. Сумма флага uses_prime была бы 6618 (число мест), вместо 5345 (число уникальных людей), что привело бы к критическому завышению показателя.

## Часть 6. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 14)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.")
print("\n--- служебная строка, не удаляйте ---")
print("HW2_ANSWERS=" + json.dumps(
    {"client_filter": CLIENT_FILTER, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 7. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–6 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: в нём две части, каждая до 1 балла сверху. Они засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется у преподавателя целиком — в том числе на контрольном сегменте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Загруженных таблиц для исследования хватает: `pay`, `subs` и `tariffs`. Считайте от них, а не от чисел, вписанных руками, — на контрольном сегменте код должен отработать без правок.

### Исследование А. Что подорожание дало выручке

С 1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽. Финансы сравнили выручку сегмента за март–июнь с январём–февралём и записали весь рост в заслугу подорожания. Руководитель просит проверить, сколько рублей в месяц на самом деле дала цена.

**Что сделать**

1. По месяцам окна посчитайте выручку и число успешных платежей — по каждому тарифу.
2. Разделите изменение выручки после 1 марта на две части. **Цена** — разница между тем, сколько стоили платежи марта–июня, и тем, сколько те же платежи стоили бы по старой цене. **Объём** — всё остальное.
3. Учтите две вещи, которые меняют число платежей в месяц без всякого подорожания. Подписка списывается раз в 30 дней, а месяцы разной длины. И в данных апреля есть дыра — вы видели её в вопросе 2.
4. Ответьте: сколько рублей в месяц приносит подорожание вашему сегменту — и насколько ошибается наивное сравнение «март–июнь против января–февраля».

In [ ]:
import pandas as pd

pay_subs = success.merge(subs[['subscription_id', 'tariff_id']], on='subscription_id', validate='m:1')

pay_subs['month'] = pd.to_datetime(pay_subs['paid_at']).dt.strftime('%Y-%m')

monthly_stats = pay_subs.groupby(['month', 'tariff_id']).agg(
    revenue=('amount', 'sum'),
    n_payments=('payment_id', 'size')
).reset_index()

print("Выручка и число платежей по месяцам и тарифам:")
display(monthly_stats.pivot(index='month', columns='tariff_id', values=['revenue', 'n_payments']))

jan_feb = pay_subs[pay_subs['month'].isin(['2026-01', '2026-02'])]
mar_jun = pay_subs[pay_subs['month'].isin(['2026-03', '2026-04', '2026-05', '2026-06'])]

days_jan_feb = 31 + 28 
days_mar_jun = 31 + 30 + 31 + 30 

rev_jan_feb = jan_feb['amount'].sum()
rev_mar_jun = mar_jun['amount'].sum()
count_mar_jun = len(mar_jun)

daily_rev_baseline = rev_jan_feb / days_jan_feb

expected_rev_mar_jun = daily_rev_baseline * days_mar_jun

price_effect_total = count_mar_jun * 50

volume_effect_total = (rev_mar_jun - price_effect_total) - expected_rev_mar_jun

price_effect_per_month = price_effect_total / 4
volume_effect_per_month = volume_effect_total / 4

naive_avg_mar_jun = rev_mar_jun / 4
naive_avg_jan_feb = rev_jan_feb / 2
naive_growth_per_month = naive_avg_mar_jun - naive_avg_jan_feb

true_growth_per_month = (rev_mar_jun - expected_rev_mar_jun) / 4

naive_error = naive_growth_per_month - true_growth_per_month

print("ИТОГИ ИССЛЕДОВАНИЯ А:")
print(f"1. Эффект ЦЕНЫ (вклад подорожания): {price_effect_per_month:.2f} руб. в месяц")
print(f"2. Эффект ОБЪЕМА (изменение числа платежей): {volume_effect_per_month:.2f} руб. в месяц")
print(f"3. Наивный прирост (март-июнь ср. минус янв-фев ср.): {naive_growth_per_month:.2f} руб. в месяц")
print(f"4. Истинный прирост (с поправкой на дни): {true_growth_per_month:.2f} руб. в месяц")
print(f"5. Ошибка наивного сравнения: {naive_error:.2f} руб. в месяц")
print("   (Наивный метод завышает успех, приписывая цене то, что связано с длиной месяцев)")

#### ✍️ Исследование А

**Вопрос:** Сколько рублей в месяц на самом деле дала цена и насколько ошибается наивное сравнение?

**Как проверяли:** Разделила платежи на периоды до/после 1 марта. Эффект цены рассчитан строго как 50 руб. умножить на число платежей в марте–июне. Эффект объема рассчитан как остаток от общего прироста, базовый уровень нормализован на количество календарных дней, чтобы сгладить разную длину месяцев.

**Что получилось:** Подорожание приносит сегменту в среднем: 104637.50 руб. в месяц. К тому же, эффект объема отрицательный (-87667.52 ₽/мес), что я бы объяснила разной длиной месяцев и пропуском данных в апреле. Наивное сравнение без поправки на дни показывает прирост 38823.25 руб./мес, ошибка на 21853.27 руб. в месяц.

ИТОГИ ИССЛЕДОВАНИЯ А:
1. Эффект ЦЕНЫ (вклад подорожания): 104637.50 руб. в месяц
2. Эффект ОБЪЕМА (изменение числа платежей): -87667.52 руб. в месяц
3. Наивный прирост (март-июнь ср. минус янв-фев ср.): 38823.25 руб. в месяц
4. Истинный прирост (с поправкой на дни): 16969.98 руб. в месяц
5. Ошибка наивного сравнения: 21853.27 руб. в месяц
   (Наивный метод завышает успех, приписывая цене то, что связано с длиной месяцев)

**Что это значит и чего не доказывает:** Финансы ошибаются, записывая весь рост на подорожание. Значительная часть "роста" является артефактом календаря (в марте-июне больше дней, чем в январе-феврале). Однако расчет не доказывает, что отток вызван именно ценой, он лишь фиксирует общее снижение числа платежей относительно календарной нормы.

### Исследование Б. Стоило ли подорожание подписчиков

Маркетинг опасается, что подорожание вызвало волну отмен, и предлагает вернуть старые цены. Проверьте на своём сегменте, есть ли для этого основания.

**Что сделать**

1. По месяцам с января по июнь посчитайте отток: сколько подписок закончилось в месяце из тех, что были активны на его начало. Данные — `subs`, колонки `started_at` и `ended_at`.
2. Сравните отток до и после 1 марта — отдельно для «Базового» (+25 %) и «Расширенного» (+12,5 %). Если бы цена отпугивала, какой тариф должен был отреагировать сильнее?
3. Посмотрите на причины отмен: как менялась доля `too_expensive` среди отмен по месяцам.
4. Вывод для маркетинга: есть ли в данных след подорожания в оттоке — и чего ваш расчёт не доказывает. Сколько у вас отмен в месяц и хватит ли этого, чтобы заметить эффект?

In [ ]:
import pandas as pd

subs_fixed = subs.copy()
subs_fixed['started_at'] = pd.to_datetime(subs_fixed['started_at']).dt.tz_localize(None)
subs_fixed['ended_at'] = pd.to_datetime(subs_fixed['ended_at']).dt.tz_localize(None)

def calc_churn(df):
    months = pd.period_range('2026-01', '2026-06', freq='M')
    results = []
    
    for m in months:
        m_start = m.to_timestamp()
        m_end = (m + 1).to_timestamp()
        
        active_start = len(df[
            (df['started_at'] < m_start) &
            ((df['ended_at'].isna()) | (df['ended_at'] >= m_start))
        ])
        
        ended_in_month = len(df[
            (df['ended_at'].notna()) &
            (df['ended_at'] >= m_start) &
            (df['ended_at'] < m_end)
        ])
        
        churn_rate = (ended_in_month / active_start * 100) if active_start > 0 else 0.0
        
        results.append({
            'month': str(m),
            'active_start': active_start,
            'ended': ended_in_month,
            'churn_rate_pct': round(churn_rate, 2)
        })
    return pd.DataFrame(results)

churn_all = calc_churn(subs_fixed)
churn_basic = calc_churn(subs_fixed[subs_fixed['tariff_id'] == 1])
churn_extended = calc_churn(subs_fixed[subs_fixed['tariff_id'] == 2])

print("Общий отток (% от активных на начало месяца):")
display(churn_all)

print("\nОтток Базового (тариф 1, подорожал на 25%):")
display(churn_basic)

print("\nОтток Расширенного (тариф 2, подорожал на 12.5%):")
display(churn_extended)

cancels = subs_fixed[subs_fixed['cancel_reason'].notna()].copy()
cancels['end_month'] = cancels['ended_at'].dt.to_period('M').astype(str)

cancel_stats = cancels.groupby(['end_month', 'cancel_reason']).size().unstack(fill_value=0)
cancel_stats['total_cancels'] = cancel_stats.sum(axis=1)
cancel_stats['too_expensive_pct'] = round((cancel_stats.get('too_expensive', 0) / cancel_stats['total_cancels']) * 100, 2)

print("\nСтатистика причин отмен по месяцам:")
display(cancel_stats[['total_cancels', 'too_expensive', 'too_expensive_pct']].loc['2026-01':'2026-06'])

basic_before = churn_basic[churn_basic['month'].isin(['2026-01', '2026-02'])]['churn_rate_pct'].mean()
basic_after = churn_basic[churn_basic['month'].isin(['2026-03', '2026-04', '2026-05', '2026-06'])]['churn_rate_pct'].mean()

ext_before = churn_extended[churn_extended['month'].isin(['2026-01', '2026-02'])]['churn_rate_pct'].mean()
ext_after = churn_extended[churn_extended['month'].isin(['2026-03', '2026-04', '2026-05', '2026-06'])]['churn_rate_pct'].mean()

print(f"\nСредний отток Базового: до {basic_before:.2f}%, после {basic_after:.2f}% (изменение: {basic_after - basic_before:+.2f} п.п.)")
print(f"Средний отток Расширенного: до {ext_before:.2f}%, после {ext_after:.2f}% (изменение: {ext_after - ext_before:+.2f} п.п.)")
print(f"Всего отмен в месяц в среднем: {int(cancels['end_month'].value_counts().mean())} чел.")

In [ ]:
print("Дополнительные вычисления для вывода:\n")

too_exp_mar_jun = cancel_stats.loc['2026-03':'2026-06', 'too_expensive_pct'].dropna()
min_too_exp = too_exp_mar_jun.min()
max_too_exp = too_exp_mar_jun.max()
avg_cancels = int(cancels['end_month'].value_counts().mean())

print(f"min_too_exp: {min_too_exp}")
print(f"max_too_exp: {max_too_exp}")
print(f"avg_cancels: {avg_cancels}")

#### ✍️ Исследование Б

**Вопрос**: Есть ли в данных след подорожания в оттоке и стоит ли возвращать старые цены?

**Как проверяли**: Рассчитала monthly churn rate для «Базового» и «Расширенного» тарифов до/после 1 марта. Также проанализировали динамику явной причины отмены too_expensive в абсолютных и относительных величинах.

**Что получилось**: Отток «Базового» тарифа остался стабильным на уровне 3.84% до и после 1 марта. Отток «Расширенного» тарифа тоже не показал роста, скорее незначительно снизился с 3.96% до 3.93%. Доля явных отмен по причине too_expensive после марта колеблется в диапазоне 18.37-26.56%. В абсолютных значениях в месяц отменяется в среднем около 82 подписок.

**Что это значит и чего не доказывает**: В данных нет волны отмен, которые коррелируют с подорожанием. Если бы клиенты уходили именно из-за цены, был бы заметен резкий всплеск оттока на «Базовом» тарифе (+25% к цене) по сравнению с «Расширенным» (+12,5%), но такой разницы нет.

Средний отток Базового: до 3.84%, после 3.84% (изменение: -0.00 п.п.)
Средний отток Расширенного: до 3.96%, после 3.93% (изменение: -0.02 п.п.)
Всего отмен в месяц в среднем: 82 чел.

**Однако этот расчет не доказывает** отсутствие негативного эффекта. Были учтены только явные отмены (с выбранной причиной), пассивный отток не учитывался. Также малая абсолютная база отмен (в среднем 82 чел./мес), что делает любые колебания процентов лишь статистическим шумом, а не значимым трендом. Возврат старых цен на основании этих данных будет необоснован.